In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdataproy26")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
base = f"abfss://raw@{storageName}.dfs.core.windows.net/datasets"
ruta_origen     = f"{base}/viajes"
ruta_checkpoint = f"{base}/_checkpoints/viajes_raw"
tabla_destino   = f"{catalogo}.bronze.viajes_raw"

In [0]:
print("Origen :", ruta_origen)
print("Destino:", tabla_destino)

Origen : abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes
Destino: catalog_ecobici_dev.bronze.viajes_raw


In [0]:
display(dbutils.fs.ls(ruta_origen))

path,name,size,modificationTime
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,ecobici_2026_07.csv,96069105,1794509747000
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_08.csv,ecobici_2026_08.csv,94785587,1794509747000
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_09.csv,ecobici_2026_09.csv,96389507,1794509748000
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,ecobici_2026_10.csv,102880217,1794509749000


In [0]:
viajes_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaLocation", ruta_checkpoint)
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load(ruta_origen)
)

In [0]:
viajes_bronze = (
    viajes_stream
    .withColumn("_archivo_origen", col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", current_timestamp())
)

In [0]:
consulta = (
    viajes_bronze.writeStream
    .option("checkpointLocation", ruta_checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(tabla_destino)
)
consulta.awaitTermination()

In [0]:
bronze = spark.table(tabla_destino)
print("Total de filas:", bronze.count())

Total de filas: 6065485


In [0]:
display(bronze.groupBy("_archivo_origen").count().orderBy("_archivo_origen"))

_archivo_origen,count
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,1493484
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_08.csv,1473683
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_09.csv,1498934
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,1599384


In [0]:
display(bronze.limit(10))

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,_rescued_data,_archivo_origen,_fecha_ingesta
M,22.0,4027652,282,30/09/2026,23:24:17,091,01/10/2026,00:00:02,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
F,32.0,3155801,375,30/09/2026,23:52:05,624,01/10/2026,00:00:03,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
F,25.0,3926144,002,30/09/2026,23:44:54,112,01/10/2026,00:00:14,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,24.0,2386167,563,30/09/2026,23:57:33,561,01/10/2026,00:00:24,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,41.0,2117323,114,30/09/2026,23:45:41,056,01/10/2026,00:00:27,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,41.0,8609505,142,30/09/2026,23:53:41,165,01/10/2026,00:00:28,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,25.0,8390874,115,30/09/2026,23:58:49,031,01/10/2026,00:00:29,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,33.0,6113724,345,30/09/2026,23:49:28,168,01/10/2026,00:00:33,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,21.0,4163572,154,30/09/2026,23:50:09,170,01/10/2026,00:00:37,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z
M,29.0,2405827,035,30/09/2026,23:54:26,005,01/10/2026,00:00:44,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z


In [0]:
# ⚠️ SOLO PARA REINICIAR LA CARGA DE VIAJES DESDE CERO
# Borra la tabla Y el checkpoint

# spark.sql(f"DROP TABLE IF EXISTS {tabla_destino}")
# dbutils.fs.rm(ruta_checkpoint, True)